# flash-attention-cuda — live demo

Runs the whole verification story on a free Colab GPU (Runtime → Change runtime type → **T4 GPU**):
the CPU algorithm tests, the hand-written CUDA kernel against the float64 reference,
the Triton rendering of the same algorithm, and a timing table of all of them vs PyTorch SDPA.

Repo: https://github.com/agupt0318/flash-attention-cuda

In [ ]:
!git clone -q https://github.com/agupt0318/flash-attention-cuda.git
%cd flash-attention-cuda
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. The algorithm, validated on CPU (no GPU involved)

In [ ]:
!make test

## 2. The CUDA kernel vs the reference
Compiles for this machine's GPU, then checks the usual hostile shapes at 5e-5.

In [ ]:
import torch
arch = 'sm_%d%d' % torch.cuda.get_device_capability()
!make test-gpu ARCH={arch}

## 3. PyTorch wrapper — parity vs SDPA, float64 as judge

In [ ]:
!python3 pytorch/test_parity.py bench

## 4. The Triton rendering of the same algorithm

In [ ]:
!python3 triton/test_parity.py

## 5. Everyone on one axis
CUDA kernel vs Triton vs SDPA, N = 512..4096.

In [ ]:
import sys, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
sys.path += ['pytorch', 'triton']
from flash_attn import flash_attention
from flash_attn_triton import flash_attention_triton

def time_ms(fn, iters=30):
    fn(); torch.cuda.synchronize()
    t0, t1 = torch.cuda.Event(True), torch.cuda.Event(True)
    t0.record()
    for _ in range(iters): fn()
    t1.record(); torch.cuda.synchronize()
    return t0.elapsed_time(t1) / iters

impls = {'cuda (ours)': flash_attention,
         'triton (ours)': flash_attention_triton,
         'torch SDPA': lambda q, k, v: F.scaled_dot_product_attention(q, k, v)}
seqs, results = [512, 1024, 2048, 4096], {n: [] for n in impls}
for N in seqs:
    q, k, v = (torch.randn(4, 8, N, 64, device='cuda') for _ in range(3))
    for name, fn in impls.items():
        results[name].append(time_ms(lambda: fn(q, k, v)))
        print(f'N={N:<5} {name:14} {results[name][-1]:7.3f} ms')

for name, ys in results.items(): plt.plot(seqs, ys, marker='o', label=name)
plt.xscale('log', base=2); plt.yscale('log')
plt.xlabel('sequence length'); plt.ylabel('ms (fwd)'); plt.legend()
plt.title('B=4 H=8 d=64, fp32'); plt.show()